In [6]:
import pandas as pd
import numpy as np
import os
import shutil

In [7]:
severity_rank = {
    "(X)": 1,
    "(NI0)": 2,
    "(CT)": 3,
    "(M03)": 4,
    "(M2)": 5,
    "(M1)": 6,
    "(CO3)": 7,
    "(TCX)": 8,
    "(F7)": 9,
    "(F6)": 10,
    "(F5)": 11,
    "(F3)": 12,
    "(F2)": 13,
    "(F1)": 14
}

In [8]:
def prepare_dataset(data_violent_filt):

    data_cleaned = data_violent_filt[[
    "sex", "age", "race",
    "priors_count", "juv_fel_count",
    "juv_misd_count", "juv_other_count","decile_score",
    "c_jail_in", "c_jail_out", "is_recid"
    ]].copy()
    data_cleaned['c_jail_in'] = pd.to_datetime(data_cleaned['c_jail_in'], format="%d/%m/%Y %H:%M")
    data_cleaned['c_jail_out'] = pd.to_datetime(data_cleaned['c_jail_out'], format="%d/%m/%Y %H:%M")
    data_cleaned['jail_time'] = (data_cleaned['c_jail_out'] - data_cleaned['c_jail_in']).dt.total_seconds() / (3600)
    data_cleaned.drop(columns=['c_jail_in', 'c_jail_out'], inplace=True)
    # list(data_cleaned['c_charge_degree'].unique())
    data_cleaned = data_cleaned.map(lambda x: severity_rank.get(x) if x in severity_rank else x)
    data_cleaned.dropna(inplace=True)
    return data_cleaned

In [27]:

def get_random_partitions(dataset, num_partitions,random_state):

    ds_random = dataset.sample(frac=1, random_state=random_state).reset_index(drop=True)
    indices = ds_random.index
    split_indices = np.array_split(indices, num_partitions)
    return [ds_random.loc[idx] for idx in split_indices]



def create_federated_dataset_random(num_clients=5, random_state=42, write=False):
    """
    Creates federated dataset partitions with optional file writing.
    
    Parameters:
    -----------
    num_clients : int
        Number of client partitions to create
    random_state : int
        Random seed for reproducibility
    write : bool
        If True, writes partitions to files. If False, only returns them.
    
    Returns:
    --------
    dict : Dictionary with keys 'test' and 'client_1' through 'client_N'
    """
    dataset = pd.read_csv('../data/cox-violent-parsed_filt.csv')
    dataset = prepare_dataset(dataset)
    
    # Get random partitions with fixed random state for reproducibility
    data_partitions = get_random_partitions(dataset, num_clients + 1, random_state)
    
    # Create dictionary to store partitions
    partitions_list = []
    test_set = data_partitions[0]
    
    for i in range(1, len(data_partitions)):
        partitions_list.append(data_partitions[i])
        
    
    # Optional: write to files
    if write:
        shutil.rmtree('../federated_data_random', ignore_errors=True)  # Deletes directory safely
        os.makedirs('../federated_data_random', exist_ok=True)
        
        test_set.to_csv('../federated_data_random/client_random_test.csv', index=False)
        
        for i in range(0, num_clients):
            partitions_list[i].to_csv(
                f'../federated_data_random/client_random_{i}.csv', 
                index=False
            )
        print(f"✓ Written {num_clients} client partitions + test set to ../federated_data_random/")
    
    return partitions_list, test_set

#### Centralized Dataset

In [18]:
data_violent_filt = pd.read_csv('../data/cox-violent-parsed_filt.csv')
data_prepared = prepare_dataset(data_violent_filt)
data_prepared.to_csv('../data/centralized_dataset.csv', index=False)


#### Decenralized Dataset

In [29]:
create_federated_dataset_random(num_clients=2, write=True)

✓ Written 2 client partitions + test set to ../federated_data_random/


([          sex  age              race  priors_count  juv_fel_count  \
  5673   Female   48             Other             0              0   
  5674     Male   32  African-American             0              0   
  5675     Male   36         Caucasian             2              0   
  5676     Male   20  African-American             1              1   
  5677     Male   23         Caucasian             1              0   
  ...       ...  ...               ...           ...            ...   
  11341    Male   42         Caucasian             1              0   
  11342    Male   51  African-American             0              0   
  11343    Male   19  African-American             2              0   
  11344    Male   39  African-American             6              0   
  11345    Male   49         Caucasian             4              0   
  
         juv_misd_count  juv_other_count  decile_score  is_recid    jail_time  
  5673                0                0             1         0 

In [5]:
from get_datasets import get_federated_datasets_random, get_federated_datasets_sensitive, get_centralized_dataset

federated_random = get_federated_datasets_random()
federated_sensitive = get_federated_datasets_sensitive()
centralized_dataset = get_centralized_dataset()

federated_random[0]

,id,name,first,last,sex,dob,age,age_cat,race,juv_fel_count,...,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,priors_count.1,event
0,8242.0,dylan welly,dylan,welly,Male,18/04/1994,22,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,4,Low,16/03/2013,Risk of Violence,5,Medium,0,0
1,6933.0,guillermo laffiteau,guillermo,laffiteau,Male,23/05/1957,58,Greater than 45,Hispanic,0,...,NaN,Risk of Recidivism,1,Low,20/06/2014,Risk of Violence,1,Low,4,0
2,9844.0,luis gonzalez,luis,gonzalez,Male,22/05/1992,23,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,2,Low,26/12/2013,Risk of Violence,4,Low,0,0
3,NaN,joseph ambers,joseph,ambers,Male,17/09/1966,49,Greater than 45,Caucasian,0,...,NaN,Risk of Recidivism,7,Medium,02/08/2014,Risk of Violence,1,Low,6,0
4,NaN,anthony nicholasi,anthony,nicholasi,Male,15/11/1993,22,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,2,Low,01/07/2014,Risk of Violence,4,Low,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3659,3849.0,jerrod moore,jerrod,moore,Male,15/11/1983,32,25 - 45,African-American,0,...,NaN,Risk of Recidivism,8,High,15/01/2013,Risk of Violence,6,Medium,12,0
3660,10599.0,manuel aguasvivas,manuel,aguasvivas,Male,06/12/1988,27,25 - 45,Caucasian,0,...,NaN,Risk of Recidivism,7,Medium,17/09/2014,Risk of Violence,6,Medium,2,0
3661,6226.0,lawrence gaston,lawrence,gaston,Male,08/04/1993,23,Less than 25,African-American,0,...,NaN,Risk of Recidivism,4,Low,26/08/2013,Risk of Violence,6,Medium,1,0
3662,NaN,eric mckenzie,eric,mckenzie,Male,15/03/1990,26,25 - 45,African-American,0,...,NaN,Risk of Recidivism,9,High,26/08/2013,Risk of Violence,10,High,0,0
